# Análisis de Distribución — Métricas de Oleaje y Viento
### Proyecto: Costas Mexicanas | Fuente: `cor_dev.silver.swell_metrics`

Este notebook realiza un análisis exploratorio de las condiciones meteoceanográficas 
registradas en las costas de México. El objetivo es visualizar la distribución estadística 
de las variables principales para apoyar la selección de modelos predictivos.

## 1. Carga de datos

Se accede a la tabla `silver.swell_metrics` del catálogo `cor_dev` (ambiente de desarrollo).
Esta tabla contiene registros horarios de viento y oleaje por costa, ya procesados y limpios.

**Variables disponibles:**
- `coast_name` — Nombre de la costa
- `wind_speed_ms` — Velocidad del viento (m/s)
- `wind_direction_deg` — Dirección del viento (°)
- `wave_height_m` — Altura de ola (m)
- `wave_direction_deg` — Dirección de ola (°)
- `wave_period_s` — Período de ola (s)

In [0]:
df = spark.table("cor_dev.silver.swell_metrics")
df.display()

## 2. Distribución por variable y costa

Se genera una gráfica de histograma por cada variable de interés.
Cada gráfica muestra la distribución de todas las costas superpuestas para facilitar su comparación.

> **Nota:** Se usa `cor_dev` para pruebas. Cambiar a `cor_project` para el análisis final con todas las costas.

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Convertir los datos a un formato que Python pueda graficar fácilmente
df_pandas = df.toPandas()

# Las 5 variables que nos interesan
variables = {
    "wind_speed_ms":      "Velocidad del Viento (m/s)",
    "wind_direction_deg": "Dirección del Viento (°)",
    "wave_height_m":      "Altura de Olas (m)",
    "wave_direction_deg": "Dirección de Olas (°)",
    "wave_period_s":      "Período de Olas (s)"
}

# Crear una gráfica por cada variable
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, (col, titulo) in enumerate(variables.items()):
    ax = axes[i]
    ax.hist(df_pandas[col].dropna(), bins=50, color="steelblue", edgecolor="white")
    ax.set_title(titulo, fontsize=13, fontweight="bold")
    ax.set_xlabel("Valor")
    ax.set_ylabel("Frecuencia")
    ax.grid(axis="y", alpha=0.3)

# Apagar el subplot sobrante
axes[5].set_visible(False)

plt.suptitle("Distribución de Variables Meteoceanográficas\nSabancuy, Campeche", 
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("distribucion_variables.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Gráfica guardada como distribucion_variables.png")

In [0]:
import pandas as pd
import matplotlib.pyplot as plt

df_pandas = df.toPandas()

# Las costas únicas en los datos
costas = df_pandas["coast_name"].unique()

# Las 5 variables que nos interesan
variables = {
    "wind_speed_ms":      "Velocidad del Viento (m/s)",
    "wind_direction_deg": "Dirección del Viento (°)",
    "wave_height_m":      "Altura de Olas (m)",
    "wave_direction_deg": "Dirección de Olas (°)",
    "wave_period_s":      "Período de Olas (s)"
}

# Una gráfica por variable, dentro de cada una una línea por costa
for col, titulo in variables.items():
    fig, ax = plt.subplots(figsize=(10, 5))

    for costa in costas:
        datos = df_pandas[df_pandas["coast_name"] == costa][col].dropna()
        ax.hist(datos, bins=50, alpha=0.5, edgecolor="white", label=costa)

    ax.set_title(f"Distribución — {titulo}", fontsize=14, fontweight="bold")
    ax.set_xlabel("Valor")
    ax.set_ylabel("Frecuencia")
    ax.legend(title="Costa")
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"distribucion_{col}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅ Guardada: distribucion_{col}.png")

## 3. Generación y exportación de gráficas

Se generan 5 histogramas interactivos, uno por cada variable meteorológica.
Cada gráfica muestra la distribución de todas las costas superpuestas para facilitar su comparación.

Las gráficas se guardan automáticamente en formato HTML en la carpeta `/tmp/graficas_distribucion/`.

**Archivos generados:**
- `wind_speed_ms.html` — Velocidad del Viento
- `wind_direction_deg.html` — Dirección del Viento
- `wave_height_m.html` — Altura de Olas
- `wave_direction_deg.html` — Dirección de Olas
- `wave_period_s.html` — Período de Olas

> Los archivos HTML son interactivos: se pueden abrir en cualquier navegador, hacer zoom y filtrar costas desde la leyenda.

In [0]:
import pandas as pd
import plotly.graph_objects as go
import os

df = spark.table("cor_dev.silver.swell_metrics")

output_dir = "/tmp/graficas_distribucion"
os.makedirs(output_dir, exist_ok=True)
print(f"✅ Carpeta creada en: {output_dir}")

df_pandas = df.toPandas()
costas = df_pandas["coast_name"].unique()

variables = {
    "wind_speed_ms":      "Velocidad del Viento (m/s)",
    "wind_direction_deg": "Dirección del Viento (°)",
    "wave_height_m":      "Altura de Olas (m)",
    "wave_direction_deg": "Dirección de Olas (°)",
    "wave_period_s":      "Período de Olas (s)"
}

for col, titulo in variables.items():
    fig = go.Figure()

    for costa in costas:
        datos = df_pandas[df_pandas["coast_name"] == costa][col].dropna()
        fig.add_trace(go.Histogram(
            x=datos,
            name=costa,
            opacity=0.6,
            nbinsx=50
        ))

    fig.update_layout(
        title=f"Distribución — {titulo}",
        xaxis_title="Valor",
        yaxis_title="Frecuencia",
        barmode="overlay",
        legend_title="Costa",
        template="plotly_white",
        width=900,
        height=500
    )

    # Solo HTML, sin PNG
    html_path = f"{output_dir}/{col}.html"
    fig.write_html(html_path)

    fig.show()
    print(f"✅ Guardada: {col}.html")

print(f"\n📁 Todos los archivos están en: {output_dir}")